In [2]:
import os
import random
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker


fake = Faker()

random.seed(42)
Faker.seed(42)

NUM_CUSTOMERS = 1000
NUM_PRODUCTS = 600
NUM_ORDERS = 2000
NUM_ORDER_ITEMS = 4000

OUTPUT_DIR = "data/raw"

CUSTOMER_TYPES = ["REGULAR", "PREMIUM", "VIP"]

CATEGORIES = {
    "Electronics": ["Mobile", "Laptop", "Audio", "Accessories"],
    "Clothing": ["Men", "Women", "Kids", "Footwear"],
    "Home": ["Kitchen", "Furniture", "Decor", "Appliances"],
    "Books": ["Technology", "Fiction", "Business", "Education"]
}

REGIONS = ["NORTH", "SOUTH", "EAST", "WEST", "CENTRAL"]

ORDER_STATUSES = [
    "PLACED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]


def random_date(start_date, end_date):
    delta = end_date - start_date
    random_days = random.randint(0, delta.days)
    random_seconds = random.randint(0, 86399)

    return start_date + timedelta(
        days=random_days,
        seconds=random_seconds
    )


def generate_customers():
    customers = []

    start_date = datetime(2024, 1, 1)
    end_date = datetime(2026, 7, 31)

    for i in range(1, NUM_CUSTOMERS + 1):

        customer_id = f"CUST{i:05d}"
        customer_name = fake.name()

        email = fake.email()

        customers.append({
            "customer_id": customer_id,
            "customer_name": customer_name,
            "email": email,
            "registration_date": random_date(
                start_date,
                end_date
            ).strftime("%Y-%m-%d"),
            "customer_type": random.choices(
                CUSTOMER_TYPES,
                weights=[70, 25, 5],
                k=1
            )[0]
        })

    df = pd.DataFrame(customers)

    invalid_count = max(1, int(NUM_CUSTOMERS * 0.02))

    invalid_indices = random.sample(
        range(NUM_CUSTOMERS),
        invalid_count
    )

    invalid_emails = [
        "invalidemail.com",
        "user@",
        "missingdomain@",
        "not-an-email",
        "customer.email.com"
    ]

    for index in invalid_indices:
        df.loc[index, "email"] = random.choice(invalid_emails)

    return df


def generate_products():
    products = []

    product_id = 1

    for category, subcategories in CATEGORIES.items():

        for subcategory in subcategories:

            for _ in range(
                NUM_PRODUCTS // sum(
                    len(values) for values in CATEGORIES.values()
                )
            ):

                name = fake.catch_phrase()

                cost_price = round(
                    random.uniform(100, 50000),
                    2
                )

                products.append({
                    "product_id": f"P{product_id:05d}",
                    "product_name": name,
                    "category": category,
                    "subcategory": subcategory,
                    "cost_price": cost_price
                })

                product_id += 1

    while len(products) < NUM_PRODUCTS:

        category = random.choice(list(CATEGORIES.keys()))
        subcategory = random.choice(CATEGORIES[category])

        products.append({
            "product_id": f"P{product_id:05d}",
            "product_name": fake.catch_phrase(),
            "category": category,
            "subcategory": subcategory,
            "cost_price": round(
                random.uniform(100, 50000),
                2
            )
        })

        product_id += 1

    df = pd.DataFrame(products[:NUM_PRODUCTS])

    messy_count = max(
        1,
        int(NUM_PRODUCTS * 0.10)
    )

    messy_indices = random.sample(
        range(NUM_PRODUCTS),
        messy_count
    )

    for index in messy_indices:

        product_name = df.loc[index, "product_name"]

        variation = random.choice([
            f"  {product_name}",
            f"{product_name}  ",
            product_name.upper(),
            product_name.lower(),
            f"  {product_name.upper()}  ",
            f" {product_name.lower()} "
        ])

        df.loc[index, "product_name"] = variation

    return df


def generate_orders(customers_df):
    orders = []

    customer_ids = customers_df["customer_id"].tolist()

    start_date = datetime(2025, 1, 1)
    end_date = datetime(2026, 7, 31)

    for i in range(1, NUM_ORDERS + 1):

        order_id = f"ORD{i:06d}"

        customer_id = random.choice(customer_ids)

        order_date = random_date(
            start_date,
            end_date
        )

        status = random.choices(
            ORDER_STATUSES,
            weights=[15, 20, 45, 10, 10],
            k=1
        )[0]

        region_code = random.choice(REGIONS)

        orders.append({
            "order_id": order_id,
            "customer_id": customer_id,
            "order_date": order_date.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "status": status,
            "region_code": region_code
        })

    df = pd.DataFrame(orders)

    missing_customer_count = max(
        1,
        int(NUM_ORDERS * 0.05)
    )

    missing_customer_indices = random.sample(
        range(NUM_ORDERS),
        missing_customer_count
    )

    for index in missing_customer_indices:
        df.loc[index, "customer_id"] = None

    wrong_date_count = max(
        1,
        int(NUM_ORDERS * 0.05)
    )

    wrong_date_indices = random.sample(
        range(NUM_ORDERS),
        wrong_date_count
    )

    for index in wrong_date_indices:

        original_date = datetime.strptime(
            df.loc[index, "order_date"],
            "%Y-%m-%d %H:%M:%S"
        )

        df.loc[index, "order_date"] = original_date.strftime(
            "%d-%m-%Y"
        )

    return df


def generate_order_items(orders_df, products_df):
    order_items = []

    order_ids = orders_df["order_id"].tolist()
    product_ids = products_df["product_id"].tolist()

    item_id = 1

    for order_id in order_ids:

        product_id = random.choice(product_ids)

        quantity = random.randint(1, 5)

        product_row = products_df[
            products_df["product_id"] == product_id
        ].iloc[0]

        cost_price = product_row["cost_price"]

        unit_price = round(
            cost_price * random.uniform(1.10, 1.80),
            2
        )

        discount_percent = round(
            random.uniform(0, 40),
            2
        )

        order_items.append({
            "item_id": f"ITEM{item_id:06d}",
            "order_id": order_id,
            "product_id": product_id,
            "quantity": quantity,
            "unit_price": unit_price,
            "discount_percent": discount_percent
        })

        item_id += 1

    remaining_items = NUM_ORDER_ITEMS - len(order_items)

    for _ in range(remaining_items):

        order_id = random.choice(order_ids)
        product_id = random.choice(product_ids)

        quantity = random.randint(1, 5)

        product_row = products_df[
            products_df["product_id"] == product_id
        ].iloc[0]

        cost_price = product_row["cost_price"]

        unit_price = round(
            cost_price * random.uniform(1.10, 1.80),
            2
        )

        discount_percent = round(
            random.uniform(0, 40),
            2
        )

        order_items.append({
            "item_id": f"ITEM{item_id:06d}",
            "order_id": order_id,
            "product_id": product_id,
            "quantity": quantity,
            "unit_price": unit_price,
            "discount_percent": discount_percent
        })

        item_id += 1

    df = pd.DataFrame(order_items)

    negative_count = max(
        1,
        int(NUM_ORDER_ITEMS * 0.03)
    )

    negative_indices = random.sample(
        range(NUM_ORDER_ITEMS),
        negative_count
    )

    for index in negative_indices:

        original_quantity = df.loc[index, "quantity"]

        df.loc[index, "quantity"] = -abs(
            original_quantity
        )

    return df


def save_data(customers_df, products_df, orders_df, order_items_df):

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    customers_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "customers.csv"
        ),
        index=False
    )

    products_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "products.csv"
        ),
        index=False
    )

    orders_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "orders.csv"
        ),
        index=False
    )

    order_items_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "order_items.csv"
        ),
        index=False
    )


def print_summary(
    customers_df,
    products_df,
    orders_df,
    order_items_df
):

    print("\n" + "=" * 60)
    print("E-COMMERCE DATA GENERATION COMPLETED")
    print("=" * 60)

    print("\nGenerated files:")
    print(f"customers.csv     : {len(customers_df)} rows")
    print(f"products.csv      : {len(products_df)} rows")
    print(f"orders.csv        : {len(orders_df)} rows")
    print(f"order_items.csv   : {len(order_items_df)} rows")

    print("\nIntentional data issues:")

    missing_customers = orders_df["customer_id"].isna().sum()

    negative_quantities = (
        order_items_df["quantity"] < 0
    ).sum()

    wrong_date_count = 0

    for value in orders_df["order_date"]:
        try:
            datetime.strptime(
                value,
                "%Y-%m-%d %H:%M:%S"
            )
        except ValueError:
            wrong_date_count += 1

    invalid_email_count = 0

    for email in customers_df["email"]:

        if (
            "@" not in email
            or "." not in email.split("@")[-1]
        ):
            invalid_email_count += 1

    messy_product_count = 0

    for name in products_df["product_name"]:

        if (
            name != name.strip()
            or name != name.title()
        ):
            messy_product_count += 1

    print(
        f"Missing customer IDs : {missing_customers}"
    )

    print(
        f"Negative quantities  : {negative_quantities}"
    )

    print(
        f"Wrong date formats   : {wrong_date_count}"
    )

    print(
        f"Invalid emails       : {invalid_email_count}"
    )

    print(
        f"Messy product names  : {messy_product_count}"
    )

    print("\nOutput directory:")
    print(os.path.abspath(OUTPUT_DIR))

    print("=" * 60)


def main():

    print("Generating customers...")
    customers_df = generate_customers()

    print("Generating products...")
    products_df = generate_products()

    print("Generating orders...")
    orders_df = generate_orders(customers_df)

    print("Generating order items...")
    order_items_df = generate_order_items(
        orders_df,
        products_df
    )

    print("Saving CSV files...")
    save_data(
        customers_df,
        products_df,
        orders_df,
        order_items_df
    )

    print_summary(
        customers_df,
        products_df,
        orders_df,
        order_items_df
    )


if __name__ == "__main__":
    main()

Generating customers...
Generating products...
Generating orders...
Generating order items...
Saving CSV files...

E-COMMERCE DATA GENERATION COMPLETED

Generated files:
customers.csv     : 1000 rows
products.csv      : 600 rows
orders.csv        : 2000 rows
order_items.csv   : 4000 rows

Intentional data issues:
Missing customer IDs : 100
Negative quantities  : 120
Wrong date formats   : 100
Invalid emails       : 20
Messy product names  : 600

Output directory:
/content/data/raw
